# Modélisation et suivi des expériences avec MLflow

Ce notebook entraîne une première baseline de scoring crédit avec une régression logistique. L'objectif est d'obtenir un résultat simple, reproductible et suivi manuellement dans MLflow.

In [1]:
from pathlib import Path

import mlflow
import pandas as pd
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, confusion_matrix, f1_score, recall_score, roc_auc_score
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

# ---------- Configuration ----------
DATA_PATH = Path("../data/processed/application_train_final_encoded.csv")
TARGET_COLUMN = "TARGET"
ID_COLUMN = "SK_ID_CURR"
TEST_SIZE = 0.20
RANDOM_STATE = 42
MODEL_PARAMS = {
    "C": 1.0,
    "solver": "lbfgs",
    "max_iter": 1000,
    "class_weight": None,
    "random_state": RANDOM_STATE,
}

mlflow.set_tracking_uri("sqlite:///../mlflow.db")
mlflow.set_experiment("credit_scoring")

2026/09/08 22:06:35 INFO mlflow.agent.hint: Load the `instrumenting-with-mlflow-tracing` skill at C:\Users\Philippe MAGNE\Documents\3 - DEV\P6_initiez_vous_au_MLOps_(partie1_sur_2)\.venv\Lib\site-packages\mlflow\assistant\skills\instrumenting-with-mlflow-tracing\SKILL.md before writing any tracing code; it ships with this MLflow install. Set MLFLOW_DISABLE_AGENT_HINT=1 to silence this.


<Experiment: artifact_location=('file:C:/Users/Philippe MAGNE/Documents/3 - '
 'DEV/P6_initiez_vous_au_MLOps_(partie1_sur_2)/notebooks/mlruns/1'), creation_time=1788896468178, effective_trace_archival_retention=None, experiment_id='1', last_update_time=1788896468178, lifecycle_stage='active', name='credit_scoring', tags={}, trace_location=None, workspace='default'>

## Chargement et validation des données

Le fichier final est rechargé dans ce notebook. La présence de l'identifiant et de la cible est contrôlée avant la modélisation.

In [2]:
# ---------- Chargement et validation ----------
if not DATA_PATH.is_file():
    raise FileNotFoundError(f"Fichier introuvable : {DATA_PATH.resolve()}")

df = pd.read_csv(DATA_PATH)
required_columns = {ID_COLUMN, TARGET_COLUMN}
missing_columns = required_columns.difference(df.columns)
if missing_columns:
    raise ValueError(f"Colonnes obligatoires absentes : {sorted(missing_columns)}")

print(f"Dataset chargé : {df.shape[0]:,} lignes et {df.shape[1]:,} colonnes")

Dataset chargé : 307,510 lignes et 321 colonnes


## Séparation entraînement-validation

`SK_ID_CURR` est exclu des variables. La stratification conserve la proportion des deux classes dans les sous-ensembles.

In [3]:
# ---------- Variables explicatives et cible ----------
X = df.drop(columns=[ID_COLUMN, TARGET_COLUMN])
y = df[TARGET_COLUMN]
X_train, X_valid, y_train, y_valid = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y
)

print(f"Entraînement : {len(X_train):,} lignes")
print(f"Validation : {len(X_valid):,} lignes")

Entraînement : 246,008 lignes
Validation : 61,502 lignes


## Prétraitement et modèle

La médiane et la standardisation sont apprises uniquement sur l'entraînement afin d'éviter une fuite de données.

In [4]:
# ---------- Prétraitement et modèle ----------
model = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("classifier", LogisticRegression(**MODEL_PARAMS)),
])

## Entraînement, évaluation et suivi MLflow

Les faux négatifs (`fn`) représentent les mauvais clients acceptés à tort ; les faux positifs (`fp`) représentent les bons clients refusés à tort. Ces deux erreurs métier sont enregistrées avec les quatre métriques générales.

In [5]:
# ---------- Entraînement et suivi MLflow ----------
with mlflow.start_run(run_name="logistic_regression_baseline"):
    mlflow.set_tag("description", "Baseline de régression logistique")
    mlflow.log_param("model_type", "LogisticRegression")
    mlflow.log_param("test_size", TEST_SIZE)
    mlflow.log_param("imputation", "median")
    mlflow.log_param("scaling", "standard")
    for parameter_name, parameter_value in MODEL_PARAMS.items():
        mlflow.log_param(parameter_name, parameter_value)

    model.fit(X_train, y_train)
    y_pred = model.predict(X_valid)
    y_score = model.predict_proba(X_valid)[:, 1]

    metrics = {
        "roc_auc": roc_auc_score(y_valid, y_score),
        "accuracy": accuracy_score(y_valid, y_pred),
        "recall": recall_score(y_valid, y_pred),
        "f1_score": f1_score(y_valid, y_pred),
    }
    for metric_name, metric_value in metrics.items():
        mlflow.log_metric(metric_name, metric_value)

    # ---------- Matrice de confusion ----------
    confusion = confusion_matrix(y_valid, y_pred)
    tn, fp, fn, tp = confusion.ravel()
    mlflow.log_metric("fn", int(fn))
    mlflow.log_metric("fp", int(fp))

    confusion_display = pd.DataFrame(
        confusion,
        index=["Réel : bon client (0)", "Réel : mauvais client (1)"],
        columns=["Prédit : bon client (0)", "Prédit : mauvais client (1)"],
    )
    display(confusion_display)
    print(f"TN — bon client correctement accepté       : {tn}")
    print(f"FP — bon client refusé à tort               : {fp}")
    print(f"FN — mauvais client accepté à tort          : {fn}")
    print(f"TP — mauvais client correctement identifié  : {tp}")

pd.Series(metrics, name="validation")

,Prédit : bon client (0),Prédit : mauvais client (1)
Réel : bon client (0),56421,116
Réel : mauvais client (1),4824,141


TN — bon client correctement accepté       : 56421
FP — bon client refusé à tort               : 116
FN — mauvais client accepté à tort          : 4824
TP — mauvais client correctement identifié  : 141


roc_auc     0.767102
accuracy    0.919677
recall      0.028399
f1_score    0.054002
Name: validation, dtype: float64